# Evidence Vacuum v2: resolution reviewer evaluation (Colab T4)

Evaluates **G-base** (Qwen3.5 4B, untuned; amendment A2) and **G-ft** (same + Adaption LoRA adapter) as *resolution reviewers* on the
**45 frozen v1 investigator answers** (`eval_inputs.jsonl`), scored against the frozen v1 labels. Pre-registered in
`01_protocol/PROTOCOL_v2_resolution_reviewer.md`.

**Setup:** Runtime → T4 GPU. Secrets: `HF_TOKEN` (optional: Qwen3.5 4B is ungated, but a token avoids rate limits).
**Drive folder** `MyDrive/evidence-vacuum-v2/` must contain: `eval_inputs.jsonl` and (for G-ft) `adapter.tar.gz` downloaded from Adaption.
Each prompt is the full reviewer input (system + user), sent as one user turn through the chat template: the same text used as the
`prompt` column in the Adaption SFT upload. Greedy decoding. Resumable after disconnects.

In [ ]:
!pip -q install "transformers==5.23.0" "peft==0.21.3" "accelerate==1.15.0"

In [ ]:
import json, os, re, tarfile, time, math
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.colab import drive, userdata
drive.mount("/content/drive")
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"
BASE = "Qwen/Qwen3.5-4B"  # amendment A2 (Llama 3.2 3B not offered for training on this key)
ROOT = "/content/drive/MyDrive/evidence-vacuum-v2"
INPUTS = f"{ROOT}/eval_inputs.jsonl"
ADAPTER_TAR, ADAPTER_DIR = f"{ROOT}/adapter.tar.gz", f"{ROOT}/adapter"
OUT = f"{ROOT}/outputs"; os.makedirs(OUT, exist_ok=True)
recs = [json.loads(l) for l in open(INPUTS) if l.strip()]
print(len(recs), "eval inputs |", torch.cuda.get_device_name(0))

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map="cuda", token=HF_TOKEN).eval()

STATUS = {"malicious_supported": "escalate", "benign_supported": "close", "insufficient_evidence": "collect_more"}
def parse(text):
    m = re.search(r"\{.*\}", text or "", flags=re.S)
    if not m: return None
    try: d = json.loads(m.group(0))
    except json.JSONDecodeError: return None
    if d.get("epistemic_status") not in STATUS or d.get("recommended_action") not in ("close","escalate","collect_more"): return None
    return d

@torch.inference_mode()
def review(m, prompt):
    enc = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True, enable_thinking=False)
    ids = enc["input_ids"].to(m.device)
    mask = enc["attention_mask"].to(m.device)
    out = m.generate(ids, attention_mask=mask, max_new_tokens=320, do_sample=False,
                     temperature=None, top_p=None, pad_token_id=tok.eos_token_id)
    text = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
    return text, parse(text), int(ids.shape[1])

def run(m, name):
    path = f"{OUT}/{name}.jsonl"
    done = {json.loads(l)["run_id"] for l in open(path)} if os.path.exists(path) else set()
    with open(path, "a") as f:
        for r in tqdm([r for r in recs if r["run_id"] not in done], desc=name):
            t0 = time.time(); text, d, n = review(m, r["prompt"])
            f.write(json.dumps({"run_id": r["run_id"], "reviewer": name, "raw": text, "parsed": d, "n_in": n,
                                "secs": round(time.time()-t0, 2)}) + "\n"); f.flush()
    return path

## 1. Smoke test (one input), then G-base on all 45

In [ ]:
t, d, n = review(model, recs[0]["prompt"]); print(n, "tokens in"); print(t[:400]); print("parsed:", d)

In [ ]:
gbase = run(model, "G-base")

## 2. G-ft: unpack the Adaption checkpoint and run the same inputs

In [ ]:
from peft import PeftModel
if not os.path.exists(f"{ADAPTER_DIR}/adapter_config.json"):
    os.makedirs(ADAPTER_DIR, exist_ok=True)
    with tarfile.open(ADAPTER_TAR) as t: t.extractall(ADAPTER_DIR, filter="data")  # rejects absolute paths, ../ traversal, links out of the dir and device files
found = [r for r, _, fs in os.walk(ADAPTER_DIR) if "adapter_config.json" in fs]; assert found, "no adapter_config.json in archive"
cfg = json.load(open(f"{found[0]}/adapter_config.json")); print("adapter base:", cfg.get("base_model_name_or_path"), "r=", cfg.get("r"))
ft = PeftModel.from_pretrained(model, found[0]).eval()
gft = run(ft, "G-ft")

## 3. Score against the frozen v1 labels (the final action is the reviewer's action; invalid → investigator's action)

In [ ]:
def mcnemar(b, c):
    n = b + c
    return 1.0 if n == 0 else min(1.0, 2 * sum(math.comb(n, i) for i in range(min(b, c) + 1)) / 2**n)
WANT = {"MALICIOUS_SUPPORTED": {"escalate"}, "BENIGN_SUPPORTED": {"close"}, "INSUFFICIENT": {"escalate", "collect_more"}}
def final(r, out):
    return out["parsed"]["recommended_action"] if out and out["parsed"] else r["investigator_action"]
res = {}
for name in ("investigator", "G-base", "G-ft"):
    outs = {} if name == "investigator" else {json.loads(l)["run_id"]: json.loads(l) for l in open(f"{OUT}/{name}.jsonl")}
    rows = []
    for r in recs:
        a = r["investigator_action"] if name == "investigator" else final(r, outs.get(r["run_id"]))
        rows.append(dict(r, action=a, invalid=(name != "investigator" and not (outs.get(r["run_id"]) or {}).get("parsed")),
                         wrong=int(a not in WANT[r["label"]]), unsupported_close=int(a == "close" and r["closure_defensible"] == "no"),
                         unnecessary_esc=(int(a != "close") if r["label"] == "BENIGN_SUPPORTED" else None)))
    res[name] = rows
    ben = [x for x in rows if x["unnecessary_esc"] is not None]; nc = [x for x in rows if x["closure_defensible"] == "no"]
    print(f"{name:12} unnecessary escalation {sum(x['unnecessary_esc'] for x in ben)}/{len(ben)} | unsupported closures "
          f"{sum(x['unsupported_close'] for x in nc)}/{len(nc)} | wrong action {sum(x['wrong'] for x in rows)}/{len(rows)} | "
          f"invalid {sum(x['invalid'] for x in rows)}")
for key in ("wrong", "unsupported_close"):
    b = sum(1 for x, y in zip(res["G-base"], res["G-ft"]) if x[key] == 1 and y[key] == 0)
    c = sum(1 for x, y in zip(res["G-base"], res["G-ft"]) if x[key] == 0 and y[key] == 1)
    print(f"G-base vs G-ft, {key}: b={b} c={c} exact McNemar p={mcnemar(b, c):.3f}")
json.dump({k: v for k, v in res.items()}, open(f"{OUT}/scored_v2.json", "w"), indent=1)
print("saved", f"{OUT}/scored_v2.json")